# 09 Capabilities: Persistence, Memory & Operations

By the end you can add checkpoints, stores, interrupts, retries, streaming, subgraphs, and time travel.

Set up the project path and reload shared helpers from this repo.


In [ ]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Pick up edits to shared/ without a full kernel restart (re-run this cell)
for _name in (
    "shared.dataflow",
    "shared.notebook_display",
    "shared.llm",
    "shared.bootcamp_fixtures",
):
    importlib.reload(importlib.import_module(_name))

print(f"Project root: {ROOT}")


Verify DeepSeek credentials before running LangGraph cells.


In [ ]:
import os

def env_status():
    keys = {
        "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
        "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "deepseek"),
        "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
        "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    }
    for k, v in keys.items():
        print(f"{k}: {v}")
    return keys

ENV = env_status()


Load the chat model helper used by live cells below.


In [ ]:
import os
from shared.llm import get_model

def require_llm():
    return get_model()


Import DATAFLOW printers and ask_approval once. These label every graph step, in later cells, read preview() lines to see state moving between nodes like spell components chaining together.


In [ ]:
from shared.dataflow import preview, print_dataflow, print_agent_dataflow, print_final_state
from shared.notebook_display import ask_approval


Checkpoint every node with InMemorySaver, triage, draft, finalize each call the model. Run once, then inspect CHECKPOINTS count and the final sent status in output.


In [ ]:
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from shared.bootcamp_fixtures import search_kb
from shared.llm import get_model

model = get_model()
TRIAGE_SYSTEM = "You triage travel support tickets. One short status line."
DRAFT_SYSTEM = "You draft travel support answers using only the provided KB facts."

class SupportState(TypedDict):
    request: str
    status: str
    reply: str

def triage(state: SupportState):
    text = model.invoke([
        SystemMessage(content=TRIAGE_SYSTEM),
        HumanMessage(content=state["request"]),
    ]).content
    return {"status": f"Triaged: {str(text).strip()[:80]}"}

def draft(state: SupportState):
    hits = search_kb(state["request"])
    text = model.invoke([
        SystemMessage(content=DRAFT_SYSTEM),
        HumanMessage(content=f"Facts: {'; '.join(hits[:2])}\nRequest: {state['request']}"),
    ]).content
    return {"reply": str(text).strip()[:300]}

def finalize(state: SupportState):
    return {"status": "sent", "reply": state["reply"]}

checkpointer = InMemorySaver()
g = StateGraph(SupportState)
g.add_node("triage", triage)
g.add_node("draft", draft)
g.add_node("finalize", finalize)
g.add_edge(START, "triage")
g.add_edge("triage", "draft")
g.add_edge("draft", "finalize")
g.add_edge("finalize", END)
persistence_graph = g.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "travel-support-42"}}
inputs = {"request": "Schengen visa stay limits?", "status": "", "reply": ""}

print_dataflow(persistence_graph, inputs, config=config)
history = list(persistence_graph.get_state_history(config))
print("\nCHECKPOINTS")
print(f"count: {len(history)}")
for i, snap in enumerate(reversed(list(history))):
    print(f" [{i}] next={snap.next} status={preview(snap.values.get('status', ''))} reply={preview(snap.values.get('reply', ''))}")


Draw persistence_graph so you see the linear triage to draft to finalize pipeline.


In [ ]:
from IPython.display import Image, display

# Show workflow
display(Image(persistence_graph.get_graph().draw_mermaid_png()))


Store long-term prefs in a namespace; load_preferences reads them via get_store(). DATAFLOW summary should mention beach budget and Star Alliance loyalty text.


In [ ]:
from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

store = InMemoryStore()
NS = ("traveler-7", "travel_preferences")
store.put(NS, "budget", {"text": "prefers beach destinations under $600"})
store.put(NS, "airline", {"text": "loyalty:Star Alliance"})
print("STORE namespace:", NS)

class PrefState(TypedDict):
    summary: str

def load_preferences(_: PrefState):
    items = get_store().search(NS, limit=10)
    return {"summary": "; ".join(i.value.get("text", "") for i in items)}

g = StateGraph(PrefState)
g.add_node("load_preferences", load_preferences)
g.add_edge(START, "load_preferences")
g.add_edge("load_preferences", END)
store_graph = g.compile(store=store)

from IPython.display import Image, display

display(Image(store_graph.get_graph().draw_mermaid_png()))
print_dataflow(store_graph, {"summary": ""})


Define hitl_graph with interrupt() inside book_trip and compile with InMemorySaver. This graph pauses until a human approves, inspect the single book_trip node.


In [ ]:
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt

class BookingState(TypedDict):
    destination: str
    budget: int
    status: str

def book_trip(state: BookingState):
    # LangGraph built-in: first call pauses; Command(resume=...) value returns here
    human_decision = interrupt({
        "action": "book_trip",
        "destination": state["destination"],
        "budget": state["budget"],
        "message": f"Approve booking {state['destination']} for ${state['budget']}?",
    })
    approved = human_decision.get("approved", False) if isinstance(human_decision, dict) else bool(human_decision)
    if approved:
        return {"status": f"BOOKED {state['destination']}"}
    return {"status": "CANCELLED"}

checkpointer = InMemorySaver()
hitl_builder = StateGraph(BookingState)
hitl_builder.add_node("book_trip", book_trip)
hitl_builder.add_edge(START, "book_trip")
hitl_builder.add_edge("book_trip", END)
hitl_graph = hitl_builder.compile(checkpointer=checkpointer)


Visualize hitl_graph, one node, one interrupt point.


In [ ]:
from IPython.display import Image, display

# Show workflow
display(Image(hitl_graph.get_graph().draw_mermaid_png()))


Stream until interrupt() pauses the run. Look for __interrupt__ in stream output and next nodes showing book_trip still pending.


In [ ]:

config = {"configurable": {"thread_id": "booking-approval-1"}}
inputs = {"destination": "Lisbon", "budget": 480, "status": ""}

print("REAL-TIME HITL, stream until interrupt()")
for chunk in hitl_graph.stream(inputs, config=config):
    print(f" stream: {preview(chunk)}")

snap = hitl_graph.get_state(config)
print(f"\nCHECKPOINT (graph paused)")
print(f" next nodes: {snap.next}")
print(f" state: {preview(snap.values)}")
print(f" interrupts: {preview(snap.interrupts[0].value if snap.interrupts else None)}")


After pause, ask_approval collects your y/n, then Command(resume=...) finishes the booking. FINAL STATE status should read BOOKED Lisbon or CANCELLED based on your answer.


In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "booking-approval-1"}}
snap = hitl_graph.get_state(config)
payload = snap.interrupts[0].value

print("REAL-TIME HITL, wait for human, then stream resume")
human_decision = ask_approval(payload)

for chunk in hitl_graph.stream(Command(resume=human_decision), config=config):
    print(f" stream: {preview(chunk)}")

print_final_state(hitl_graph.get_state(config).values, keys=["destination", "budget", "status"])


RetryPolicy retries flaky_lookup until success, simulates transient API failures. DATAFLOW should list three attempts before quote appears.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy

class TransientAPIError(Exception):
    pass

_calls = {"count": 0}
_attempt_log: list[str] = []

def flaky_lookup(city: str) ->  str:
    _calls["count"] += 1
    _attempt_log.append(f"attempt {_calls['count']}")
    if _calls["count"] < 3:
        raise TransientAPIError("503 upstream")
    return f"{city}:flights from $420"

class FTState(TypedDict):
    city: str
    quote: str

g = StateGraph(FTState)
g.add_node("lookup", lambda s: {"quote": flaky_lookup(s["city"])}, retry=RetryPolicy(max_attempts=4, initial_interval=0.01, retry_on=TransientAPIError))
g.add_edge(START, "lookup")
g.add_edge("lookup", END)
retry_graph = g.compile()

from IPython.display import Image, display

display(Image(retry_graph.get_graph().draw_mermaid_png()))
print("DATAFLOW (retry attempts)")
print_dataflow(retry_graph, {"city": "Lisbon", "quote": ""})
print(" attempts:", ": ".join(_attempt_log))


Stream updates and custom events from LLM nodes via get_stream_writer(). Compare DATAFLOW (updates) node lines with custom: event lines, two lenses on one run.


In [ ]:
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer
from shared.bootcamp_fixtures import search_kb
from shared.llm import get_model

model = get_model()
DRAFT_SYSTEM = "You answer travel policy questions in one sentence using provided facts."

class StreamState(TypedDict):
    request: str
    step: str
    reply: str

def triage(state: StreamState):
    writer = get_stream_writer()
    writer({"event": "triage_started", "request_preview": state["request"]})
    return {"step": "triaged"}

def draft(state: StreamState):
    writer = get_stream_writer()
    hits = search_kb(state["request"])
    text = model.invoke([
        SystemMessage(content=DRAFT_SYSTEM),
        HumanMessage(content=f"Facts: {'; '.join(hits[:2])}\nQuestion: {state['request']}"),
    ]).content
    writer({"event": "draft_ready"})
    return {"reply": str(text).strip(), "step": "drafted"}

checkpointer = InMemorySaver()
g = StateGraph(StreamState)
g.add_node("triage", triage)
g.add_node("draft", draft)
g.add_edge(START, "triage")
g.add_edge("triage", "draft")
g.add_edge("draft", END)
stream_graph = g.compile(checkpointer=checkpointer)
config = {"configurable": {"thread_id": "stream-demo-1"}}

stream_inputs = {"request": "Schengen question", "step": "", "reply": ""}

print("DATAFLOW (updates)")
print_dataflow(stream_graph, stream_inputs, config=config)

print("\nDATAFLOW (custom events)")
for event in stream_graph.stream(
    stream_inputs,
    config={**config, "configurable": {**config["configurable"], "thread_id": "stream-demo-2"}},
    stream_mode="custom",
):
    print(f" custom: {event}")


Draw stream_graph, same triage/draft shape as persistence but instrumented for streaming.


In [ ]:
from IPython.display import Image, display

# Show workflow
display(Image(stream_graph.get_graph().draw_mermaid_png()))


Pattern B2: wrapper subgraph: parent and child use different state keys, so run_research maps request to topic and returns only facts. Two DATAFLOW blocks: parent run_research/compose_reply and child lookup_policy.


In [ ]:
import operator
from typing import Annotated, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from shared.bootcamp_fixtures import search_kb
from shared.llm import get_model

model = get_model()
RESEARCH_SYSTEM = "Summarize travel policy facts in one short phrase."
COMPOSE_SYSTEM = "Compose a one-sentence customer answer using the research facts."

class ResearchState(TypedDict):
    topic: str
    facts: Annotated[list[str], operator.add]

def lookup_policy(state: ResearchState):
    hits = search_kb(state["topic"])
    summary = model.invoke([
        SystemMessage(content=RESEARCH_SYSTEM),
        HumanMessage(content="; ".join(hits[:2])),
    ]).content
    return {"facts": [str(summary).strip()[:120]]}

research_builder = StateGraph(ResearchState)
research_builder.add_node("lookup_policy", lookup_policy)
research_builder.add_edge(START, "lookup_policy")
research_builder.add_edge("lookup_policy", END)
research_subgraph = research_builder.compile()

class ParentState(TypedDict):
    request: str
    facts: Annotated[list[str], operator.add]
    reply: str

def run_research(state: ParentState):
    sub_out = research_subgraph.invoke({"topic":state["request"], "facts": []})
    return {"facts": sub_out["facts"]}

def compose_reply(state: ParentState):
    text = model.invoke([
        SystemMessage(content=COMPOSE_SYSTEM),
        HumanMessage(content="; ".join(state["facts"])),
    ]).content
    return {"reply": f"Answer using: {str(text).strip()}"}

parent = StateGraph(ParentState)
parent.add_node("run_research", run_research)
parent.add_node("compose_reply", compose_reply)
parent.add_edge(START, "run_research")
parent.add_edge("run_research", "compose_reply")
parent.add_edge("compose_reply", END)

parent_graph = parent.compile()
sub_inputs = {"request": "Schengen stay limits", "facts": [], "reply": ""}
print("DATAFLOW (parent graph)")
print_dataflow(parent_graph, sub_inputs)
print("\nDATAFLOW (research subgraph only)")
print_dataflow(research_subgraph, {"topic":sub_inputs["request"], "facts": []})


Render parent and research subgraphs side by side. A compiled create_agent() could be invoked the same way, wrapper isolation, not shared messages.


In [ ]:
from IPython.display import Image, display

# Show workflow (parent_graph)
display(Image(parent_graph.get_graph().draw_mermaid_png()))

# Show workflow (research_subgraph)
display(Image(research_subgraph.get_graph().draw_mermaid_png()))


**Time travel** lets you rewind a persisted thread, edit state at an earlier checkpoint, and resume from there.

Three ideas to keep straight:

1. **`thread_id`** — checkpoints are grouped per thread (same `config` as earlier cells).
2. **`get_state_history(config)`** — snapshots after each node; read them to see where you are in the run.
3. **`update_state(..., as_node="step_a")`** — rewind to the checkpoint *after* `step_a`, patch fields, then **`invoke(None, fork_config)`** to re-run downstream nodes.

The toy graph below is `step_a` (mark searched) → `step_b` (draft using `destination`). We run it for Lisbon, rewind to `step_a`, change destination to Porto, and resume so `step_b` drafts for Porto.


In [ ]:
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

class TTState(TypedDict):
    destination: str
    status: str

g = StateGraph(TTState)
g.add_node("step_a", lambda s: {"status": "searched"})
g.add_node("step_b", lambda s: {"status": f"drafted for {s['destination']}"})
g.add_edge(START, "step_a")
g.add_edge("step_a", "step_b")
g.add_edge("step_b", END)
time_travel_graph = g.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "time-travel-1"}}
inputs = {"destination": "Lisbon", "status": ""}

display(Image(time_travel_graph.get_graph().draw_mermaid_png()))


**Step 1 — Run once and read the checkpoint trail.** Each row is state saved after a node. Notice `step_b` reads `destination` from the prior checkpoint.


In [ ]:
final = time_travel_graph.invoke(inputs, config)
print("FINAL STATE:", final)

history = list(time_travel_graph.get_state_history(config))
print(f"\nCHECKPOINTS ({len(history)} snapshots, oldest first):")
for i, snap in enumerate(reversed(history)):
    v = snap.values
    print(
        f"  [{i}] next={snap.next} "
        f"destination={v.get('destination', '')!r} status={v.get('status', '')!r}"
    )


**Step 2 — Rewind, patch, resume.** `update_state` only edits the forked snapshot (you may still see the old `status` until downstream nodes re-run). Pass the returned `fork_config` to `invoke(None, ...)` so `step_b` runs again with Porto.


In [ ]:
fork_config = time_travel_graph.update_state(
    config,
    {"destination": "Porto"},
    as_node="step_a",
)
forked = time_travel_graph.get_state(fork_config).values
print("FORKED (patched, before resume):", forked)
print("  note: status still says Lisbon until step_b re-runs")

resumed = time_travel_graph.invoke(None, fork_config)
print("RESUMED (after step_b re-runs):", resumed)


Recap: you can now add checkpoints, stores, interrupts, retries, streaming, subgraphs, and time travel. Open the next notebook when ready, 10, Deep Agents Harness.


In [ ]:
print("You can now: add checkpoints, stores, interrupts, retries, streaming, subgraphs, and time travel")
print("Next: 10 Deep Agents Harness")
